# Skill extraction rules (spaCy EntityRuler)


TIER 1 - PHRASES (any case)  
        Long / distinctive names. "projektledning", "Projektledning" and "PROJEKTLEDNING" all match. This is the default tier.  
        Example: "Google Cloud Platform", "Bluetooth-teknik".
 
TIER 2 - EXACT CASE (short or ambiguous words)
    Short acronyms and words that are also ordinary words.
    Only the exact spelling matches. "SAP" matches, "sap" does not.  
    Example: "R", "SAP", "REST", "Node", "Dart".
 
TIER 3 - CUSTOM ALIASES (synonyms -> one canonical skill)
    Different spellings / languages / abbreviations that should all map to ONE canonical name. Each alias is declared as any-case or exact-case, because "Machine learning" is safe in any case but "ML" and "AI" are not.


In [1]:
import re
import spacy
 
LABEL = "SKILL"
 

# TIER 1 - PHRASES, matched in ANY case
# Taxonomy labels, copied as-is. Labels with "/" or "(...)" are expanded into aliases automatically.

TIER1_PHRASES = [
    "Bower",
    "Bluetooth-teknik",
    "Microsoft Autopilot",
    "Office SharePoint Server (MOSS)",
    "Ledningssystem (ISO 9000-serien)",
    "WAP-teknik",
    "Asset Information Management",
    "Sun Java Enterprise System",
    "IBM Integration Bus",
    "Elixir",
    "AZ-900: Microsoft Azure Fundamentals",
    "Web Content Accessibility Guidelines/WCAG",
    "Systemförvaltning",
    "Glassfish Server",
    "Visual Studio",
    "Cybersäkerhet",
    "Citrix",
    "Informationsmodellering",
    "Grafiska gränssnitt/GUI",
    "Molnteknik",
    "CoffeeScript",
    "Nätverksteknik",
    "Praktisk ProjektStyrning/PPS",
    "Oracle Certified Professional/OCP",
    "Software AG WebMethods",
    "Datalog",
    "UNIX",
    "Enterprise JavaBeans",
    "Local Interconnect Network",
    "Webbutveckling",
    "3G-teknik",
    "Weblogic Server",
    "Vagrant",
    "Geografiskt informationssystem/GIS",
    "Scrum",
    "SVEA",
    "IBM Cloud",
    "Google Cloud Platform",
    "AZ-204: Developing Solutions for Microsoft Azure",
    "Visual Studio Code",
    "Oracle NetSuite",
    "IBM WebSphere",
    "LINQ",
    "MB-600: Microsoft Dynamics 365 Power Platform Solution Architect",
    "Oracle Fusion Middleware",
    "Windows",
    "Verksamhetsanalys",
    "Länkad data",
    "RFID-teknik",
    "Budgetansvar",
    "Databasdesign",
    "Windows Azure",
    "TCP/IP",
    "API-design",
    "Oracle Certified Database Administrator",
    "SOAP",
    "Masterdata",
    "Amazon Web Services (AWS)",
    "Förändringsledning",
    "Windows Server",
    "Solaris",
    "Microsoft Certified: Azure Administrator Associate",
    "BlackBerry",
    "Automationsteknik",
    "Systemutredning",
    "Ubuntu",
    "VxWorks",
    "Object Component Model/COM",
    "Projektledning",
    "Apache Tomcat",
    "Eclipselink",
    "Medieteknik",
    "Windows Powershell",
    "Entity Framework Core",
    "Systemintegrering",
    "IBM MQ",
    "Prediktiv analys",
    "Objektorienterad metodik",
    "Agila arbetsmetoder",
    "Blockkedjeteknik",
    "Parallell programmering",
    "Microsoft Biztalk",
    "ERP-system",
    "TypeScript",
    "Informationsarkitektur",
    "MDX",
    "Kravhantering",
    "OpenUP",
    "Kvalitetskontrollsystem",
    "Android",
    "Configuration Management/CM",
    "Oracle Data Integrator",
    "Infor",
    "Software AG Integration Suite",
    "OpenJPA",
    "PHP Developer Certification",
    "JBoss Application Server",
    "LDAP",
    "Vyper",
    "OS/390",
    "Docker",
    "VM/CMS",
    "Microsoft Certified System Engineer/MCSE",
    "Clojurescript",
    "Unified Modeling Language",
    "Java Server Pages (JSP)",
    "Microsoft Intune",
    "Oracle JCAPS",
    "XML Web Services",
    "CASE-verktyg",
    "Informationsstrategi",
    "IP-telefoniteknik",
    "Oracle Service Bus",
    "Scaled Agile Framework",
    "Microsoft Certified Database Administrator/MCDBA",
    "5G-teknik",
    "Hadoop",
    "Datawarehouse",
    "Microsoft Dynamics 365",
    "ISAC",
    "IR-teknik",
    "Certified JavaScript Developer",
    "XQuery",
    "Apache Kafka",
    "Nätverksdesign",
    "Corba",
    "OCaml",
    "IANA/ICANN",
    "Design Thinking",
    "Microsoft 365 Certified: Fundamentals",
    "Tjänsteorienterande arkitektur/SOA",
    "Rational Rose",
    "Felsökningsverktyg",
    "Distribuerad objektteknik",
    "Säkerhetsanalys",
    "Salesforce Cloud",
    "Testautomatisering/Test automation",
    "Toplink",
    "Människa-datorinteraktion/MDI",
    "Oracle Certified Associate",
    "OS/400",
    "Telematik",
    "IBM Domino",
    "4G-teknik",
    "Hibernate",
    "Windows för mobila enheter",
    "Model-view-viewmodel (MVVM)",
    "Informatik",
    "Mac OS X",
    "ASP.NET Core",
    "Associate Android Developer",
    "Oracle Certified Database Operator/DBO",
    "Microsoft Certified Professional/MCP",
    "Microsoft Certified Application Developer/MCAD",
    "Öppen källkod/Open Source",
    "Microsoft Certified Solution Developer/MCSD",
    "Web Accessibility Initiative/WAI",
    "ASP.NET MVC",
    "JBoss Enterprise Middleware Suite (JEMS)",
    "IT-konsultuppdrag",
    "Dataskyddsförordningen",
    "PROPS",
    "Clojure",
    "Datamodellering",
    "PowerScript",
    "OpenEdge Advanced Business Language",
    "Linux",
    "IRIX",
    "Interaktionsdesign",
]
 

# TIER 2 - EXACT CASE, short or ambiguous words
# {canonical skill: [exact spellings]}

TIER2_EXACT_CASE = {
    "R": ["R"],
    "DM": ["DM"],                 
    "OOS": ["OOS"],              
    "Odoo": ["Odoo"],
    "OOAD": ["OOAD"],
    "OSE": ["OSE"], 
    "LSE": ["LSE"], 
    "SAP": ["SAP"],
    "REST": ["REST"],
    "Dos": ["Dos"],        
    "IOS": ["IOS", "iOS"],
    "Node": ["Node"],
    "Logic": ["Logic"],
    "Ethics": ["Ethics"],
    "Dart": ["Dart"],
    "Grunt": ["Grunt"],
    "SPARQL": ["SPARQL"],
    "A+": ["A+"],
    "SCCM": ["SCCM"],
    "TIBCO": ["TIBCO"],
    "RUP": ["RUP"],
    "MVS": ["MVS"],
    "pm3": ["pm3"],
    "GNU": ["GNU"],
    "VMS": ["VMS"],
    "IIS": ["IIS"],
    "pSOS": ["pSOS"],
    "N1QL": ["N1QL"],
    "VSE": ["VSE"],
    "JCL": ["JCL"],
    "Dataskyddsförordningen": ["GDPR"],
}
 

# TIER 3 - CUSTOM ALIASES
# {canonical skill: {"any_case": [...], "exact_case": [...]}}

TIER3_CUSTOM = {
    "Maskininlärning": {
        "any_case": ["Machine learning", "Maskininlärning", "Maskin inlärning"],
        "exact_case": ["ML"],
    },
    "Artificiell intelligens": {
        "any_case": ["Artificial Intelligence", "Artificiell intelligens"],
        "exact_case": ["AI"],
    },
    "Enterprise JavaBeans": {
        "any_case": ["Enterprise JavaBeans", "ramverk/EJB"],
        "exact_case": ["EJB"],
    },
    "Java Persistence API": {
        "any_case": ["Java Persistence API"],
        "exact_case": ["JPA"],
    },
    "Hibernate": {
        "any_case": ["Hibernate"],
        "exact_case": [],
    },
    "Local Interconnect Network": {
        "any_case": ["Local Interconnect Network"],
        "exact_case": ["LIN"],
    },
    "Scaled Agile Framework": {
        "any_case": ["Scaled Agile Framework"],
        "exact_case": ["SAFe"],
    },
}

In [5]:
# Compile the three tiers into two lookups: {form: canonical}
any_case, exact_case = {}, {}

for label in TIER1_PHRASES:
    parts = [p.strip() for p in re.split(r"[/()]", label) if p.strip()]
    for p in parts:
        if len(parts) > 1 and p.isupper():   # abbreviation such as AWS or GIS
            exact_case[p] = label
        else:
            any_case[p] = label

for canonical, forms in TIER2_EXACT_CASE.items():
    for f in forms:
        exact_case[f] = canonical

for canonical, groups in TIER3_CUSTOM.items():
    for f in groups.get("any_case", []):
        any_case[f] = canonical
    for f in groups.get("exact_case", []):
        exact_case[f] = canonical

# Two rulers: any-case (LOWER) first, exact-case (literal text) second
nlp = spacy.blank("sv")

def add_ruler(name, rules, lower):
    config = {"phrase_matcher_attr": "LOWER"} if lower else {}
    ruler = nlp.add_pipe("entity_ruler", name=name, config=config)
    ruler.add_patterns(
        [{"label": "SKILL", "pattern": form, "id": canon} for form, canon in rules.items()]
    )

add_ruler("any_case", any_case, lower=True)
add_ruler("exact_case", exact_case, lower=False)

# Test
text = "Jag kan projektledning, GIS, AWS, ML och GDPR. Jag köpte 2 ml mjölk och sap."
for ent in nlp(text).ents:
    print(f"{ent.text:<15} -> {ent.ent_id_}")

projektledning  -> Projektledning
GIS             -> Geografiskt informationssystem/GIS
AWS             -> Amazon Web Services (AWS)
ML              -> Maskininlärning
GDPR            -> Dataskyddsförordningen
